<a href="https://colab.research.google.com/github/tbooonmak-collab/my-team/blob/karn-dev/Japanese_Restaurant_POS_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍣 ระบบร้านอาหารญี่ปุ่น (Japanese Restaurant POS Simulation)
### โปรเจกต์จำลองระบบรับออเดอร์ คำนวณราคา และวิเคราะห์ข้อมูล

ในส่วนนี้เราจะสร้าง Class หลัก 3 ตัว ได้แก่ `MenuItem` (เมนู), `Table` (โต๊ะ), และ `Order` (ออเดอร์) เพื่อจำลองการเปิดโต๊ะ สั่งอาหารรวดเดียว แล้วปิดบิลทันที

## ส่วนที่ 4: แปลงข้อมูล บันทึกลง CSV และ SQLite
เพื่อทำคะแนน Bonus ในส่วนของ SQL เราจะแยกข้อมูลเป็น 2 ตาราง:
1. `orders_df`: ข้อมูลหัวบิล (โต๊ะ, จำนวนคน, ยอดรวม)
2. `order_items_df`: ข้อมูลรายละเอียดอาหารในบิลแต่ละใบ (สั่งอะไรบ้าง)

In [1]:
# แปลง Object เป็น List of Dictionaries เพื่อทำ Pandas DataFrame
orders_data = []
order_items_data = []

for o in all_orders:
    # 1. ข้อมูลหัวบิล
    orders_data.append({
        "order_id": o.order_id,
        "table_number": o.table.table_number,
        "num_customers": o.table.num_customers,
        "subtotal": o.calculate_subtotal(),
        "vat": o.calculate_vat(),
        "grand_total": o.calculate_grand_total(),
        "status": o.status
    })

    # 2. ข้อมูลรายละเอียดเมนูที่สั่งในบิลนั้น
    for item in o.items:
        order_items_data.append({
            "order_id": o.order_id,
            "menu_name": item['menu'].name,
            "price_per_unit": item['menu'].price,
            "quantity": item['qty'],
            "total_item_price": item['menu'].price * item['qty']
        })

# สร้าง DataFrame
df_orders = pd.DataFrame(orders_data)
df_order_items = pd.DataFrame(order_items_data)

# บันทึกเป็น CSV
df_orders.to_csv("restaurant_orders.csv", index=False)
df_order_items.to_csv("restaurant_order_items.csv", index=False)
print("💾 บันทึกไฟล์ CSV สำเร็จ!")

# สร้างฐานข้อมูล SQLite และ Insert ข้อมูลลงไป 2 ตาราง
conn = sqlite3.connect("japanese_restaurant.db")
df_orders.to_sql("orders", conn, if_exists="replace", index=False)
df_order_items.to_sql("order_items", conn, if_exists="replace", index=False)
print("🗄️ บันทึกข้อมูลลง SQLite (2 ตาราง) ")

NameError: name 'all_orders' is not defined

## ส่วนที่ 5: วิเคราะห์ข้อมูลด้วย Pandas (Data Analysis)
สำรวจข้อมูลเบื้องต้นและตอบคำถามเชิงธุรกิจ (เช่น ยอดขายรวม เมนูขายดี ยอดเฉลี่ยต่อโต๊ะ)

In [ ]:
df_analysis = pd.read_csv("restaurant_orders.csv")
df_items_analysis = pd.read_csv("restaurant_order_items.csv")

print("--- ข้อมูลเบื้องต้นของบิล (df.info) ---")
df_analysis.info()

print("\n--- สถิติเบื้องต้น (df.describe) ---")
display(df_analysis.describe())

# 1. สรุปยอดขายรวมทั้งหมดของร้าน
total_revenue = df_analysis['grand_total'].sum()
print(f"\n💰 ยอดขายสุทธิรวม (รวม VAT): {format_currency(total_revenue)}")

# 2. จัดอันดับเมนูขายดี (ดูจากจำนวนจานที่ขายได้ และ รายได้จากเมนูนั้น)
print("\n🏆 จัดอันดับเมนูขายดี (เรียงตามรายได้):")
menu_summary = df_items_analysis.groupby('menu_name').agg(
    total_qty=('quantity', 'sum'),
    total_revenue=('total_item_price', 'sum')
).reset_index()

# เรียงลำดับเมนูทำเงินสูงสุด
top_menus = menu_summary.sort_values(by='total_revenue', ascending=False)
display(top_menus)

# 3. วิเคราะห์ยอดลูกค้าต่อโต๊ะ
avg_pax = df_analysis['num_customers'].mean()
avg_bill = df_analysis['grand_total'].mean()
print(f"\n👥 จำนวนลูกค้าเฉลี่ยต่อโต๊ะ: {avg_pax:.1f} คน")
print(f"🧾 ยอดใช้จ่ายเฉลี่ยต่อบิล: {format_currency(avg_bill)}")